# Exercise 1 — one free node, bar **forces** given

Four fixed points `p1(0,0,0)`, `p2(5,0,3)`, `p3(0,7,3)`, `p4(7,5,0)` hold one free point `p0`.
Every bar carries a force of 1 kN. Gravity at `p0` is 5 kN.

- Determine `p0`.
- Calculate the sum of forces in node `p0`.
- What do you observe?

## 1. The net

In [1]:
import numpy as np

# connectivity: rows = bars a, b, c, d   columns = nodes p0..p4
C = np.array([
    [1, -1,  0,  0,  0],
    [1,  0, -1,  0,  0],
    [1,  0,  0, -1,  0],
    [1,  0,  0,  0, -1],
])

free  = [0]            # p0
fixed = [1, 2, 3, 4]   # p1..p4

C_N = C[:, free]
C_F = C[:, fixed]

# fixed node coordinates (p1..p4)
x_F = np.array([0, 5, 0, 7])
y_F = np.array([0, 0, 7, 5])
z_F = np.array([0, 3, 3, 0])

# load on the free node: gravity 5 kN, pointing down
p_x = np.array([0])
p_y = np.array([0])
p_z = np.array([-5])

# bar forces, one per bar [kN]
f = np.array([1, 1, 1, 1])

## 2. Determine p0

The method needs force densities $q = f / l$, but the lengths $l$ are unknown until `p0` is known.
So: guess `q`, solve, measure the lengths, correct `q = f / l`, repeat.

In [2]:
q = np.array([1, 1, 1, 1])     # first guess

for step in range(10):
    Q = np.diag(q)

    # equation 6.32
    A = C_N.T @ Q @ C_N
    x_N = np.linalg.solve(A, p_x - C_N.T @ Q @ C_F @ x_F)
    y_N = np.linalg.solve(A, p_y - C_N.T @ Q @ C_F @ y_F)
    z_N = np.linalg.solve(A, p_z - C_N.T @ Q @ C_F @ z_F)

    # bar lengths
    x = np.concatenate([x_N, x_F])
    y = np.concatenate([y_N, y_F])
    z = np.concatenate([z_N, z_F])
    l = np.sqrt((C @ x)**2 + (C @ y)**2 + (C @ z)**2)

    print("step", step, " p0 =", np.round([x_N[0], y_N[0], z_N[0]], 2), " bar forces =", np.round(q * l, 2))

    # correct the force densities so that the bar forces become 1 kN
    q = f / l

step 0  p0 = [3.   3.   0.25]  bar forces = [4.25 4.53 5.71 4.48]
step 1  p0 = [ 3.12  2.74 -4.46]  bar forces = [1.43 1.8  1.6  1.41]
step 2  p0 = [ 3.1   2.81 -7.77]  bar forces = [1.45 1.38 1.31 1.42]
step 3  p0 = [  3.08   2.88 -11.3 ]  bar forces = [1.37 1.3  1.27 1.35]
step 4  p0 = [  3.06   2.92 -15.38]  bar forces = [1.32 1.27 1.26 1.32]
step 5  p0 = [  3.05   2.94 -20.26]  bar forces = [1.3  1.26 1.25 1.3 ]
step 6  p0 = [  3.04   2.96 -26.23]  bar forces = [1.28 1.25 1.25 1.28]
step 7  p0 = [  3.03   2.97 -33.56]  bar forces = [1.27 1.25 1.24 1.27]
step 8  p0 = [  3.02   2.97 -42.65]  bar forces = [1.27 1.25 1.24 1.27]
step 9  p0 = [  3.02   2.98 -53.95]  bar forces = [1.26 1.25 1.24 1.26]


`p0` never settles. It falls lower at every step and the bar forces never reach 1 kN.

## 3. Sum of forces in node p0

Each bar pulls `p0` towards its fixed point with 1 kN. Add the four pulls and the load.

In [3]:
# pull of each bar on p0 = force * direction (direction = unit vector from p0 to the fixed point)
pull_x = f * -(C @ x) / l
pull_y = f * -(C @ y) / l
pull_z = f * -(C @ z) / l

sum_x = pull_x.sum() + p_x[0]
sum_y = pull_y.sum() + p_y[0]
sum_z = pull_z.sum() + p_z[0]

print("sum of forces in p0 =", np.round([sum_x, sum_y, sum_z], 2), "kN")

sum of forces in p0 = [-0.    0.   -1.01] kN


## 4. What do you observe?

- The sum of forces is not zero: about 1 kN is still pulling down. Four bars of 1 kN can lift 4 kN at most, the load is 5 kN. **There is no equilibrium point `p0` for these forces.**
- With given **forces** the problem is non-linear: the bar directions and lengths depend on the unknown `p0`. It needs iterations, and a solution may not exist.
- With given **force densities** (Exercise 2) the same net is solved in one linear step.

Try `p_z = np.array([-1])` in the first cell: the loop then settles at `p0 = (3.14, 2.60, 0.24)` and the sum of forces becomes zero.